In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Clone the repo
!git clone https://github.com/yeonsumia/palmistry.git /content/palmistry_model
!pip install -r /content/palmistry_model/code/requirements.txt --quiet
!pip install pillow-heif fpdf2 groq --quiet

# 2. Download landmarker model
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task -O /content/palmistry_model/code/hand_landmarker.task

# 3. Patch rectification.py
new_rectification_code = '''
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

WARP_SUCCESS = 1
MODEL_PATH = "/content/palmistry_model/code/hand_landmarker.task"

_base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
_options = vision.HandLandmarkerOptions(
    base_options=_base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    running_mode=vision.RunningMode.IMAGE
)
_landmarker = vision.HandLandmarker.create_from_options(_options)


def warp_image(path_to_image, path_to_warped_image):
    pts_index = list(range(21))
    pts_target_normalized = np.float32([[1-0.48203104734420776, 0.9063420295715332],
                                        [1-0.6043621301651001, 0.8119394183158875],
                                        [1-0.6763232946395874, 0.6790258884429932],
                                        [1-0.7340714335441589, 0.5716733932495117],
                                        [1-0.7896472215652466, 0.5098430514335632],
                                        [1-0.5655680298805237, 0.5117031931877136],
                                        [1-0.5979393720626831, 0.36575648188591003],
                                        [1-0.6135331392288208, 0.2713503837585449],
                                        [1-0.6196483373641968, 0.19251111149787903],
                                        [1-0.4928809702396393, 0.4982593059539795],
                                        [1-0.4899863600730896, 0.3213786780834198],
                                        [1-0.4894656836986542, 0.21283167600631714],
                                        [1-0.48334982991218567, 0.12900274991989136],
                                        [1-0.4258815348148346, 0.5180916786193848],
                                        [1-0.4033462107181549, 0.3581996262073517],
                                        [1-0.3938145041465759, 0.2616880536079407],
                                        [1-0.38608720898628235, 0.1775170862674713],
                                        [1-0.36368662118911743, 0.5642163157463074],
                                        [1-0.33553171157836914, 0.44737303256988525],
                                        [1-0.3209102153778076, 0.3749568462371826],
                                        [1-0.31213682889938354, 0.3026996850967407]])

    image = cv2.flip(cv2.imread(path_to_image), 1)
    image_height, image_width, _ = image.shape

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    result = _landmarker.detect(mp_image)

    if not result.hand_landmarks:
        return None

    hand_landmarks = result.hand_landmarks[0]
    pts = np.float32([[hand_landmarks[i].x*image_width, hand_landmarks[i].y*image_height] for i in pts_index])
    pts_target = np.float32([[x*image_width, y*image_height] for x,y in pts_target_normalized])
    M, mask = cv2.findHomography(pts, pts_target, cv2.RANSAC, 5.0)
    warped_image = cv2.warpPerspective(image, M, (image_width, image_height), borderMode=cv2.BORDER_REPLICATE)
    cv2.imwrite(path_to_warped_image, warped_image)
    return WARP_SUCCESS


def warp(path_to_input_image, path_to_warped_image):
    if path_to_input_image[-4:] in ["heic", "HEIC"]:
        path_to_input_image = path_to_input_image[:-4] + "jpg"
    warp_result = warp_image(path_to_input_image, path_to_warped_image)
    if warp_result is None:
        return None
    else:
        return WARP_SUCCESS
'''
with open('/content/palmistry_model/code/rectification.py', 'w') as f:
    f.write(new_rectification_code)

# 4. Patch classification.py
with open('/content/palmistry_model/code/classification.py', 'r') as f:
    content = f.read()
old_line = "skel_img = cv2.cvtColor(skeletonize(palmline_img), cv2.COLOR_BGR2GRAY)"
new_line = "skel_img = (skeletonize(palmline_img).astype(np.uint8) * 255)\n    skel_img = cv2.cvtColor(skel_img, cv2.COLOR_BGR2GRAY)"
if old_line in content:
    content = content.replace(old_line, new_line)
    with open('/content/palmistry_model/code/classification.py', 'w') as f:
        f.write(content)

# 5. Patch measurement.py
new_measurement_code = '''
from PIL import Image, ImageDraw
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

MODEL_PATH = "/content/palmistry_model/code/hand_landmarker.task"

_base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
_options = vision.HandLandmarkerOptions(
    base_options=_base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    running_mode=vision.RunningMode.IMAGE
)
_landmarker = vision.HandLandmarker.create_from_options(_options)


def measure(path_to_warped_image_mini, lines):
    image = cv2.flip(cv2.imread(path_to_warped_image_mini), 1)
    image_height, image_width, _ = image.shape

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    result = _landmarker.detect(mp_image)

    if not result.hand_landmarks:
        return None, None

    hand_landmarks = result.hand_landmarks[0]

    zero = hand_landmarks[0].y
    one = hand_landmarks[1].y
    five = hand_landmarks[5].x
    nine = hand_landmarks[9].x
    thirteen = hand_landmarks[13].x

    heart_thres_x = image_width * (1 - (nine + (five - nine) * 2 / 5))
    head_thres_x = image_width * (1 - (thirteen + (nine - thirteen) / 3))
    life_thres_y = image_height * (one + (zero - one) / 3)

    im = Image.open(path_to_warped_image_mini)
    width = 3
    if (None in lines) or (len(lines) < 3):
        return None, None
    else:
        draw = ImageDraw.Draw(im)
        heart_line = lines[0]
        head_line = lines[1]
        life_line = lines[2]

        heart_line_points = [tuple(reversed(l[:2])) for l in heart_line]
        heart_line_tip = heart_line_points[0]
        heart_content_1 = "Love line governs all matters of the heart, including romance, friendship, and commitment."
        heart_content_2 = "Your Heart line is long, which means you will have long partnership with whom you love or care." if heart_line_tip[0] < heart_thres_x else "Your Heart line is short, which means you will meet various people and have a broad range of relationships throughout your life."
        draw.line(heart_line_points, fill="red", width=width)

        head_line_points = [tuple(reversed(l[:2])) for l in head_line]
        head_line_tip = head_line_points[-1]
        head_content_1 = "Head line tells us about our intellectual curiosities and pursuits."
        head_content_2 = "Your Head line is long, which means you will explore a broad range of topics throughout your life." if head_line_tip[0] > head_thres_x else "Your Head line is short, which means you will be fascinated by one topic and dig deep into it."
        draw.line(head_line_points, fill="green", width=width)

        life_line_points = [tuple(reversed(l[:2])) for l in life_line]
        life_line_tip = life_line_points[-1]
        life_content_1 = "Life line reveals your experiences, vitality, and zest. Be careful, it has nothing to do with how long you will live!"
        life_content_2 = "Your Life line is long, which means you tend to solve problems with other people rather than by yourself." if life_line_tip[1] > life_thres_y else "Your Life line is short, which means you are independent and autonomous."
        draw.line(life_line_points, fill="blue", width=width)

        contents = [heart_content_1, heart_content_2, head_content_1, head_content_2, life_content_1, life_content_2]
        return im, contents
'''
with open('/content/palmistry_model/code/measurement.py', 'w') as f:
    f.write(new_measurement_code)

print("Repo cloned and all patches applied successfully")

Cloning into '/content/palmistry_model'...
remote: Enumerating objects: 262, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 262 (delta 10), reused 4 (delta 4), pack-reused 238 (from 3)
Receiving objects: 100% (262/262), 308.92 MiB | 34.95 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (91/91), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.3 MB/s eta 0:00:00
Repo cloned and all patches applied successfully


In [3]:
# Copy and extract the hand dataset
!cp "/content/drive/MyDrive/palmistry_tarot_project/hands-and-palm-images-dataset.zip" /content/
!unzip -q /content/hands-and-palm-images-dataset.zip -d /content/

import pandas as pd
import os

hands_dir = "/content/Hands/Hands"
print("Folder exists:", os.path.exists(hands_dir))
print("Total images:", len(os.listdir(hands_dir)))

hand_info = pd.read_csv('/content/HandInfo.csv')
print(hand_info['aspectOfHand'].value_counts())

palmar_files = hand_info[hand_info['aspectOfHand'].isin(['palmar left', 'palmar right'])]['imageName'].tolist()
print("\nTotal palmar (palm-side) images:", len(palmar_files))

Folder exists: True
Total images: 11076
aspectOfHand
dorsal right    2892
palmar right    2813
dorsal left     2788
palmar left     2583
Name: count, dtype: int64

Total palmar (palm-side) images: 5396


In [4]:
import zipfile, json, os

if not os.path.exists('/content/tarot_data'):
    with zipfile.ZipFile('/content/drive/MyDrive/palmistry_tarot_project/tarot-json.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/tarot_data')

with open('/content/tarot_data/tarot-images.json', 'r') as f:
    tarot_json = json.load(f)

df_tarot = pd.DataFrame(tarot_json['cards'])
print("Tarot cards loaded:", df_tarot.shape)

# Confirm card images exist and check the 'img' field matches actual files
print(df_tarot[['name', 'img']].head())
print("\nCards image folder exists:", os.path.exists('/content/tarot_data/cards'))
print("Sample image files:", os.listdir('/content/tarot_data/cards')[:5])

Tarot cards loaded: (78, 16)
                 name      img
0            The Fool  m00.jpg
1        The Magician  m01.jpg
2  The High Priestess  m02.jpg
3         The Empress  m03.jpg
4         The Emperor  m04.jpg

Cards image folder exists: True
Sample image files: ['m19.jpg', 'p04.jpg', 'w09.jpg', 'm07.jpg', 'p05.jpg']


In [5]:
import sys
sys.path.append('/content/palmistry_model/code')

import torch, os, shutil
from model import UNet
from rectification import warp
from detection import detect
from classification import classify
from measurement import measure
from tools import remove_background, resize

# Load model once
net = UNet(n_channels=3, n_classes=1)
net.load_state_dict(torch.load('/content/palmistry_model/code/checkpoint/checkpoint_aug_epoch70.pth', map_location=torch.device('cpu')))
net.eval()
print("Palm model loaded")

def process_one_hand(filename, hands_dir='/content/Hands/Hands', work_dir='/content/palmistry_model/code'):
    os.chdir(work_dir)
    src = os.path.join(hands_dir, filename)
    dst = f'input/{filename}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)

    resize_value = 256
    path_to_input_image = dst
    path_to_clean_image = 'results/palm_without_background.jpg'
    path_to_warped_image = 'results/warped_palm.jpg'
    path_to_warped_image_clean = 'results/warped_palm_clean.jpg'
    path_to_warped_image_mini = 'results/warped_palm_mini.jpg'
    path_to_warped_image_clean_mini = 'results/warped_palm_clean_mini.jpg'
    path_to_palmline_image = 'results/palm_lines.png'

    try:
        remove_background(path_to_input_image, path_to_clean_image)
        warp_result = warp(path_to_input_image, path_to_warped_image)
        if warp_result is None:
            return {'filename': filename, 'status': 'failed_warp', 'image_path': None}

        remove_background(path_to_warped_image, path_to_warped_image_clean)
        resize(path_to_warped_image, path_to_warped_image_clean, path_to_warped_image_mini, path_to_warped_image_clean_mini, resize_value)

        detect(net, path_to_warped_image_clean, path_to_palmline_image, resize_value)
        lines = classify(path_to_palmline_image)
        im, contents = measure(path_to_warped_image_mini, lines)

        if im is None:
            return {'filename': filename, 'status': 'failed_lines', 'image_path': None}

        os.makedirs('results/annotated', exist_ok=True)
        saved_image_path = f'{work_dir}/results/annotated/{filename}'
        im.save(saved_image_path)

        return {'filename': filename, 'status': 'success', 'contents': contents, 'image_path': saved_image_path}
    except Exception as e:
        return {'filename': filename, 'status': f'error: {str(e)[:100]}', 'image_path': None}

# Quick test
test_result = process_one_hand(palmar_files[3])  # Hand_0000041.jpg - known good from earlier testing
print(test_result)

Palm model loaded
{'filename': 'Hand_0000041.jpg', 'status': 'error: OpenCV(5.0.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.emp', 'image_path': None}


In [6]:
import os
os.chdir('/content/palmistry_model/code')
print("input/ exists:", os.path.exists('input'))
print("results/ exists:", os.path.exists('results'))

# Create them if missing
os.makedirs('input', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Check tools.py to see what remove_background actually does
with open('tools.py', 'r') as f:
    print(f.read())

input/ exists: True
results/ exists: False
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from pillow_heif import register_heif_opener

def heic_to_jpeg(heic_dir, jpeg_dir):
    register_heif_opener()  
    image = Image.open(heic_dir)
    image.save(jpeg_dir, "JPEG")

def remove_background(jpeg_dir, path_to_clean_image):
    if jpeg_dir[-4:] in ['heic', 'HEIC']:
        heic_to_jpeg(jpeg_dir, jpeg_dir[:-4] + 'jpg')
        jpeg_dir = jpeg_dir[:-4] + 'jpg'
    img = cv2.imread(jpeg_dir)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 80], dtype="uint8")
    upper = np.array([50, 255, 255], dtype="uint8")
    mask = cv2.inRange(hsv, lower, upper)
    result = cv2.bitwise_and(img, img, mask=mask)
    b, g, r = cv2.split(result)  
    filter = g.copy()
    ret, mask = cv2.threshold(filter, 10, 255, 1)
    img[mask == 255] = 255
    cv2.imwrite(path_to_clean_image, img)

def resize(path_to_warped_image, path_to_warped_image_c

In [7]:
os.chdir('/content/palmistry_model/code')

test_result = process_one_hand(palmar_files[3])
print(test_result)

{'filename': 'Hand_0000041.jpg', 'status': 'success', 'contents': ['Love line governs all matters of the heart, including romance, friendship, and commitment.', 'Your Heart line is short, which means you will meet various people and have a broad range of relationships throughout your life.', 'Head line tells us about our intellectual curiosities and pursuits.', 'Your Head line is short, which means you will be fascinated by one topic and dig deep into it.', 'Life line reveals your experiences, vitality, and zest. Be careful, it has nothing to do with how long you will live!', 'Your Life line is short, which means you are independent and autonomous.'], 'image_path': '/content/palmistry_model/code/results/annotated/Hand_0000041.jpg'}


In [9]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# Quick test
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello and confirm you're working."}]
)
print(response.choices[0].message.content)

Hello! I'm here and working—ready to help you with whatever you need.


In [10]:
import random

def draw_tarot_cards(df_tarot, n_cards=3, seed=None):
    """
    Draws n_cards randomly, assigns orientation, and includes the image path
    for each card so we can display the actual card photo later.
    """
    if seed is not None:
        random.seed(seed)

    drawn_indices = random.sample(range(len(df_tarot)), n_cards)
    drawn_cards = []

    for idx in drawn_indices:
        card = df_tarot.iloc[idx].to_dict()
        card['orientation'] = random.choice(['upright', 'reversed'])
        card['image_path'] = f"/content/tarot_data/cards/{card['img']}"
        drawn_cards.append(card)

    return drawn_cards

# Test: draw a Past/Present/Future spread
spread = draw_tarot_cards(df_tarot, n_cards=3, seed=42)

position_labels = ["Past", "Present", "Future"]
for card, label in zip(spread, position_labels):
    print(f"{label}: {card['name']} ({card['orientation']}) -> image: {card['image_path']}")
    print("Image exists:", os.path.exists(card['image_path']))

Past: Temperance (upright) -> image: /content/tarot_data/cards/m14.jpg
Image exists: True
Present: The Empress (upright) -> image: /content/tarot_data/cards/m03.jpg
Image exists: True
Future: King of Cups (upright) -> image: /content/tarot_data/cards/c14.jpg
Image exists: True


In [11]:
import json

def generate_structured_interpretation(palm_row_contents, tarot_spread, client):
    """
    palm_row_contents: the 'contents' list from process_one_hand result
    tarot_spread: list of 3 cards with position order [Past, Present, Future]
    """
    heart_text = palm_row_contents[0] + ' ' + palm_row_contents[1]
    head_text = palm_row_contents[2] + ' ' + palm_row_contents[3]
    life_text = palm_row_contents[4] + ' ' + palm_row_contents[5]

    palm_facts = f"Heart line: {heart_text}\nHead line: {head_text}\nLife line: {life_text}"

    position_labels = ["Past", "Present", "Future"]
    tarot_facts = ""
    for card, label in zip(tarot_spread, position_labels):
        meaning_pool = card['meanings']['light'] if card['orientation'] == 'upright' else card['meanings']['shadow']
        tarot_facts += f"{label}: {card['name']} ({card['orientation']}) - keywords: {', '.join(card['keywords'])}; meanings: {', '.join(meaning_pool[:3])}\n"

    prompt = f"""You are a spiritual guide and personality analyst. Based on the palm and tarot facts below, respond with ONLY a valid JSON object (no markdown, no extra text) with these exact keys:

{{
  "narrative_reading": "a 150-200 word flowing personalized reading combining palm and tarot",
  "personality_traits": ["3-4 short trait words/phrases"],
  "strengths": ["2-3 short strength phrases"],
  "growth_areas": ["2-3 short gentle growth-area phrases, framed constructively"],
  "past_description": "1-2 sentences describing what the Past tarot card reveals about this person's past",
  "present_description": "1-2 sentences describing what the Present tarot card reveals about their current life phase",
  "future_description": "1-2 sentences describing what the Future tarot card suggests is ahead for them"
}}

PALM READING FACTS:
{palm_facts}

TAROT SPREAD FACTS:
{tarot_facts}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=700
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


def generate_recommendations(structured_result, client):
    prompt = f"""You are a life coach. Based on this personality profile, respond with ONLY a valid JSON object (no markdown) with these exact keys:

{{
  "personal_growth": "one specific actionable recommendation (1 sentence)",
  "relationship_guidance": "one specific relationship recommendation (1 sentence)",
  "career_suggestion": "one specific career suggestion (1 sentence)",
  "goal_alignment": "one suggestion for aligning daily actions with long-term goals (1 sentence)"
}}

Traits: {', '.join(structured_result['personality_traits'])}
Strengths: {', '.join(structured_result['strengths'])}
Growth areas: {', '.join(structured_result['growth_areas'])}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=400
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


def generate_life_trend_analysis(structured_result, client):
    prompt = f"""You are a life-trend analyst. Based on this profile, respond with ONLY a valid JSON object (no markdown) with these exact keys:

{{
  "life_path_theme": "short phrase capturing the overarching life path theme",
  "opportunity": "one specific opportunity this person is well-positioned for (1 sentence)",
  "potential_challenge": "one constructive challenge to be mindful of (1 sentence)",
  "growth_potential": "one sentence on their growth trajectory"
}}

Traits: {', '.join(structured_result['personality_traits'])}
Strengths: {', '.join(structured_result['strengths'])}
Present life phase: {structured_result['present_description']}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=400
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

print("All 3 AI functions defined")

All 3 AI functions defined


In [17]:
ai_result = generate_structured_interpretation(test_result['contents'], spread, client)
import pprint
pprint.pprint(ai_result)

{'future_description': 'The King of Cups suggests a future of wise, diplomatic '
                       'leadership, offering calm support to others in '
                       'challenging times.',
 'growth_areas': ['balance work and rest',
                  'share knowledge openly',
                  'embrace collaborative learning'],
 'narrative_reading': 'Your short Heart line hints at a life rich with varied '
                      'connections, drawing you toward many kinds of '
                      'relationships that keep your emotional world vibrant. '
                      'Coupled with a short Head line, you tend to dive deeply '
                      'into one passion at a time, mastering its nuances '
                      'before moving on, which gives you a reputation as a '
                      'focused scholar or artist. The brief Life line '
                      'underscores your fierce independence; you thrive when '
                      'you can chart your own c

In [15]:
def generate_structured_interpretation_debug(palm_row_contents, tarot_spread, client):
    heart_text = palm_row_contents[0] + ' ' + palm_row_contents[1]
    head_text = palm_row_contents[2] + ' ' + palm_row_contents[3]
    life_text = palm_row_contents[4] + ' ' + palm_row_contents[5]

    palm_facts = f"Heart line: {heart_text}\nHead line: {head_text}\nLife line: {life_text}"

    position_labels = ["Past", "Present", "Future"]
    tarot_facts = ""
    for card, label in zip(tarot_spread, position_labels):
        meaning_pool = card['meanings']['light'] if card['orientation'] == 'upright' else card['meanings']['shadow']
        tarot_facts += f"{label}: {card['name']} ({card['orientation']}) - keywords: {', '.join(card['keywords'])}; meanings: {', '.join(meaning_pool[:3])}\n"

    prompt = f"""You are a spiritual guide and personality analyst. Based on the palm and tarot facts below, respond with ONLY a valid JSON object (no markdown, no extra text) with these exact keys:

{{
  "narrative_reading": "a 150-200 word flowing personalized reading combining palm and tarot",
  "personality_traits": ["3-4 short trait words/phrases"],
  "strengths": ["2-3 short strength phrases"],
  "growth_areas": ["2-3 short gentle growth-area phrases, framed constructively"],
  "past_description": "1-2 sentences describing what the Past tarot card reveals about this person's past",
  "present_description": "1-2 sentences describing what the Present tarot card reveals about their current life phase",
  "future_description": "1-2 sentences describing what the Future tarot card suggests is ahead for them"
}}

PALM READING FACTS:
{palm_facts}

TAROT SPREAD FACTS:
{tarot_facts}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=700
    )

    raw = response.choices[0].message.content
    print("=== RAW RESPONSE (repr) ===")
    print(repr(raw))
    print("=== finish_reason ===")
    print(response.choices[0].finish_reason)
    return raw

raw_output = generate_structured_interpretation_debug(test_result['contents'], spread, client)

=== RAW RESPONSE (repr) ===
''
=== finish_reason ===
length


In [16]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello and confirm you're working."}],
    max_tokens=500,
    reasoning_effort="low"
)
print(repr(response.choices[0].message.content))
print(response.choices[0].finish_reason)

"Hello! I'm here and ready to help. Let me know what you need."
stop


In [18]:
recommendations = generate_recommendations(ai_result, client)
trend = generate_life_trend_analysis(ai_result, client)

print("=== RECOMMENDATIONS ===")
pprint.pprint(recommendations)
print("\n=== LIFE TREND ===")
pprint.pprint(trend)

=== RECOMMENDATIONS ===
{'career_suggestion': 'Leverage your intense curiosity by pursuing roles in '
                      'research or innovation labs where you can independently '
                      'explore and then disseminate findings.',
 'goal_alignment': 'Each morning, write down the top three actions that '
                   'directly move you toward your long‑term vision and review '
                   'them before ending the day.',
 'personal_growth': 'Schedule a 15‑minute daily wind‑down ritual that includes '
                    'gentle stretching and reflection to balance work and '
                    'rest.',
 'relationship_guidance': 'Set a weekly check‑in with close friends or partner '
                          'where you actively share a recent insight or '
                          'learning to deepen connection.'}

=== LIFE TREND ===
{'growth_potential': 'Your blend of emotional adaptability and self‑nurturing '
                     'will propel you toward inc

In [20]:
def generate_structured_interpretation(palm_row_contents, tarot_spread, client):
    heart_text = palm_row_contents[0] + ' ' + palm_row_contents[1]
    head_text = palm_row_contents[2] + ' ' + palm_row_contents[3]
    life_text = palm_row_contents[4] + ' ' + palm_row_contents[5]

    palm_facts = f"Heart line: {heart_text}\nHead line: {head_text}\nLife line: {life_text}"

    position_labels = ["Past", "Present", "Future"]
    tarot_facts = ""
    for card, label in zip(tarot_spread, position_labels):
        meaning_pool = card['meanings']['light'] if card['orientation'] == 'upright' else card['meanings']['shadow']
        tarot_facts += f"{label}: {card['name']} ({card['orientation']}) - keywords: {', '.join(card['keywords'])}; meanings: {', '.join(meaning_pool[:3])}\n"

    prompt = f"""You are a spiritual guide and personality analyst. Based on the palm and tarot facts below, respond with ONLY a valid JSON object (no markdown, no extra text) with these exact keys:

{{
  "narrative_reading": "a 150-200 word flowing personalized reading combining palm and tarot",
  "personality_traits": ["3-4 short trait words/phrases"],
  "strengths": ["2-3 short strength phrases"],
  "growth_areas": ["2-3 short gentle growth-area phrases, framed constructively"],
  "past_description": "1-2 sentences describing what the Past tarot card reveals about this person's past",
  "present_description": "1-2 sentences describing what the Present tarot card reveals about their current life phase",
  "future_description": "1-2 sentences describing what the Future tarot card suggests is ahead for them"
}}

PALM READING FACTS:
{palm_facts}

TAROT SPREAD FACTS:
{tarot_facts}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1500,
        reasoning_effort="low"
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


def generate_recommendations(structured_result, client):
    prompt = f"""You are a life coach. Based on this personality profile, respond with ONLY a valid JSON object (no markdown) with these exact keys:

{{
  "personal_growth": "one specific actionable recommendation (1 sentence)",
  "relationship_guidance": "one specific relationship recommendation (1 sentence)",
  "career_suggestion": "one specific career suggestion (1 sentence)",
  "goal_alignment": "one suggestion for aligning daily actions with long-term goals (1 sentence)"
}}

Traits: {', '.join(structured_result['personality_traits'])}
Strengths: {', '.join(structured_result['strengths'])}
Growth areas: {', '.join(structured_result['growth_areas'])}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=800,
        reasoning_effort="low"
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


def generate_life_trend_analysis(structured_result, client):
    prompt = f"""You are a life-trend analyst. Based on this profile, respond with ONLY a valid JSON object (no markdown) with these exact keys:

{{
  "life_path_theme": "short phrase capturing the overarching life path theme",
  "opportunity": "one specific opportunity this person is well-positioned for (1 sentence)",
  "potential_challenge": "one constructive challenge to be mindful of (1 sentence)",
  "growth_potential": "one sentence on their growth trajectory"
}}

Traits: {', '.join(structured_result['personality_traits'])}
Strengths: {', '.join(structured_result['strengths'])}
Present life phase: {structured_result['present_description']}

Respond with ONLY the JSON object:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=800,
        reasoning_effort="low"
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

print("All 3 functions updated with new model + fix")

All 3 functions updated with new model + fix


In [21]:
ai_result = generate_structured_interpretation(test_result['contents'], spread, client)
pprint.pprint(ai_result)

{'future_description': 'The King of Cups suggests you will embody calm wisdom, '
                       'using diplomatic grace to guide and comfort others.',
 'growth_areas': ['allowing spontaneity beyond focus',
                  'balancing independence with collaboration',
                  'sharing wisdom openly'],
 'narrative_reading': 'Your short Heart line tells of a life rich with varied '
                      'connections, a tapestry of friendships and romances '
                      'that continuously refresh your emotional world. A '
                      'focused, short Head line shows you dive deeply into a '
                      'single passion, mastering its nuances and turning '
                      'curiosity into expertise. The brief Life line marks you '
                      'as fiercely independent, charting your own course with '
                      'confidence. In the past, Temperance guided you to blend '
                      'opposing forces, creating ha

In [22]:
generate_complete_pdf_report(
    user_name="Leela",
    palm_result=test_result,
    tarot_spread=spread,
    ai_result=ai_result,
    recommendations=recommendations,
    trend=trend,
    output_filename='/content/drive/MyDrive/palmistry_tarot_project/final_result.pdf'
)

Saved: /content/drive/MyDrive/palmistry_tarot_project/final_result.pdf
